In [1]:
# !pip install vllm
# !nvidia-smi
# !python --version

In [2]:
import torch

print("torch:", torch.__version__)
print("torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("compute capability:", torch.cuda.get_device_capability(0))

torch: 2.13.0+cu130
torch CUDA: 13.0
CUDA available: True
GPU: Tesla T4
compute capability: (7, 5)


In [3]:
from pynvml import (
    nvmlInit,
    nvmlShutdown,
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetName,
    nvmlDeviceGetMemoryInfo,
)

nvmlInit()
h = nvmlDeviceGetHandleByIndex(0)
m = nvmlDeviceGetMemoryInfo(h)

print("NVML GPU:", nvmlDeviceGetName(h))
print("VRAM total MB:", m.total / 1024**2)
print("VRAM used MB:", m.used / 1024**2)
print("VRAM free MB:", m.free / 1024**2)

nvmlShutdown()

NVML GPU: Tesla T4
VRAM total MB: 15360.0
VRAM used MB: 451.1875
VRAM free MB: 14908.8125


In [61]:
import importlib.metadata as m

for package in ["torch", "vllm", "aiohttp", "nvidia-ml-py"]:
    try:
        print(package, m.version(package))
    except m.PackageNotFoundError:
        print(package, "NOT INSTALLED")

torch 2.13.0
vllm 0.29.0
aiohttp 3.13.5
nvidia-ml-py 13.595.45


In [5]:
import torch
import importlib.metadata as m

print("torch imported:", torch.__version__)
print("torch metadata:", m.version("torch"))
print("torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(i, torch.cuda.get_device_name(i),
          torch.cuda.get_device_capability(i))

for package in ["vllm", "aiohttp", "nvidia-ml-py"]:
    print(package, m.version(package))

torch imported: 2.13.0+cu130
torch metadata: 2.13.0
torch CUDA: 13.0
CUDA available: True
GPU count: 2
0 Tesla T4 (7, 5)
1 Tesla T4 (7, 5)
vllm 0.29.0
aiohttp 3.13.5
nvidia-ml-py 13.595.45


In [6]:
!python -m pip check

bigframes 2.39.0 requires google-cloud-bigquery-storage, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage, which is not installed.
grpcio-tools 1.71.2 has requirement protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6.
ydata-profiling 4.18.4 has requirement numba<0.63,>=0.60, but you have numba 0.65.0.
google-colab 1.0.0 has requirement jupyter-server==2.14.0, but you have jupyter-server 2.12.5.
google-colab 1.0.0 has requirement pandas==2.2.2, but you have pandas 2.3.3.
dask-cuda 26.2.0 has requirement cuda-core==0.3.*, but you have cuda-core 1.2.0.
dopamine-rl 4.1.2 has requirement gym<=0.25.2, but you have gym 0.26.2.
moviepy 1.0.3 has requirement decorator<5.0,>=4.0.2, but you have decorator 5.3.1.
pylibcudf-cu12 26.2.1 has requirement cuda-python<13.0,>=12.9.2, but you have cuda-python 13.4.1.
numba-cuda 0.22.2 has requirement cuda-core<1.0.0,>=0.3.2, but you have cuda-core 1.2.0.
libcuml-cu12 26.2.0 has requirement cuda-toolkit[cublas,cufft,cur

In [62]:
import os
import subprocess

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"

log_file = open("/kaggle/working/vllm_server.log", "w")

server_process = subprocess.Popen(
    [
        "vllm",
        "serve",
        "Qwen/Qwen2.5-7B-Instruct-AWQ",
        "--host",
        "127.0.0.1",
        "--port",
        "8000",
    ],
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

print("PID:", server_process.pid)

PID: 792


In [79]:
!tail -n 50 /kaggle/working/vllm_server.log

(EngineCore pid=700) INFO 09-18 00:54:11 [kernel_warmup.py:124] JIT kernel warmup starting.
(EngineCore pid=700) INFO 09-18 00:54:11 [kernel_warmup.py:134] JIT kernel warmup finished in 0.00s.
Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:04<00:00,  8.73it/s]
(EngineCore pid=700) INFO 09-18 00:54:25 [model_runner.py:960] Graph capturing finished in 14 secs, took 0.22 GiB
(EngineCore pid=700) INFO 09-18 00:54:25 [gpu_worker.py:797] CUDA graph pool memory: 0.22 GiB (actual), 0.42 GiB (estimated), difference: 0.19 GiB (86.8%).
(EngineCore pid=700) INFO 09-18 00:54:25 [gpu_worker.py:860] Free memory on device (14.46/14.56 GiB) on startup. Desired GPU memory utilization is (0.92, 13.4 GiB). Actual usage is 5.56 GiB for consumed memory (weights + non-torch), 0.67 GiB for peak activation, and 0.22 GiB for CUDAGraph memory. Replace gpu_memory_utilization config with `--kv-cache-memory=7299003454` (6.8 GiB) to fit into requested memory, or `--kv-cache-memory=8439917056` (7.86 GiB) to

In [80]:
!curl -sS http://127.0.0.1:8000/v1/models

{"object":"list","data":[{"id":"Qwen/Qwen2.5-7B-Instruct-AWQ","object":"model","created":1789692879,"owned_by":"vllm","root":"Qwen/Qwen2.5-7B-Instruct-AWQ","parent":null,"max_model_len":32768,"permission":[{"id":"modelperm-b07f703c242fac41","object":"model_permission","created":1789692879,"allow_create_engine":false,"allow_sampling":true,"allow_logprobs":true,"allow_search_indices":false,"allow_view":true,"allow_fine_tuning":false,"organization":"*","group":null,"is_blocking":false}]}]}

In [27]:
%%bash

curl -sS http://127.0.0.1:8000/v1/chat/completions \
  -H "Content-Type: application/json" \
  -d '{
    "model": "Qwen/Qwen2.5-7B-Instruct-AWQ",
    "messages": [
      {
        "role": "user",
        "content": "Trả lời đúng một từ: thủ đô của Việt Nam là gì?"
      }
    ],
    "temperature": 0,
    "max_tokens": 16
  }' | python -m json.tool

{
    "id": "chatcmpl-bde4200bf8192170",
    "object": "chat.completion",
    "created": 1789689205,
    "model": "Qwen/Qwen2.5-7B-Instruct-AWQ",
    "choices": [
        {
            "index": 0,
            "message": {
                "role": "assistant",
                "content": "H\u00e0 N\u1ed9i",
                "refusal": null,
                "annotations": null,
                "audio": null,
                "function_call": null,
                "reasoning": null
            },
            "logprobs": null,
            "finish_reason": "stop",
            "stop_reason": null,
            "token_ids": null,
            "routed_experts": null
        }
    ],
    "service_tier": null,
    "system_fingerprint": "vllm-0.29.0-3e2a96bd",
    "usage": {
        "prompt_tokens": 44,
        "total_tokens": 48,
        "completion_tokens": 4,
        "prompt_tokens_details": null,
        "completion_tokens_details": null
    },
    "prompt_logprobs": null,
    "prompt_token_ids": n

In [32]:
!mkdir -p /kaggle/working/experiments/runs
!mkdir -p /kaggle/working/experiments/results

In [34]:
%%writefile /kaggle/working/experiments/benchmark_harness.py
from __future__ import annotations

import argparse
import asyncio
import importlib.metadata as metadata
import json
import os
import subprocess
import time
import uuid
from datetime import datetime, timezone
from pathlib import Path
from typing import AsyncIterator

import aiohttp
import torch
from pynvml import (
    nvmlDeviceGetHandleByIndex,
    nvmlDeviceGetMemoryInfo,
    nvmlDeviceGetName,
    nvmlInit,
    nvmlShutdown,
    nvmlSystemGetDriverVersion,
)

EXPECTED_VLLM = "0.29.0"

SUMMARY_COLUMNS = [
    "lane",
    "workload",
    "concurrency",
    "gpu",
    "n_runs",
    "ttft_p50",
    "tpot_p50",
    "tpot_p95",
    "e2e_p50",
    "alpha",
    "tau",
    "vram_peak_mb",
    "throughput",
    "speedup_vs_vanilla",
    "variance_flag",
    "commit",
    "timestamp",
]


def package_version(name: str) -> str | None:
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


def git_commit() -> str | None:
    repo_root = Path(__file__).resolve().parents[1]
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=repo_root,
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except (subprocess.CalledProcessError, FileNotFoundError):
        return None


def environment_info(gpu_index: int) -> tuple[dict, object]:
    nvmlInit()
    handle = nvmlDeviceGetHandleByIndex(gpu_index)
    name = nvmlDeviceGetName(handle)
    driver = nvmlSystemGetDriverVersion()
    mem = nvmlDeviceGetMemoryInfo(handle)

    info = {
        "gpu": name.decode() if isinstance(name, bytes) else str(name),
        "gpu_index": gpu_index,
        "gpu_memory_total_mb": round(mem.total / 1024**2, 2),
        "driver_version": driver.decode() if isinstance(driver, bytes) else str(driver),
        "torch_version": torch.__version__,
        "torch_cuda_version": torch.version.cuda,
        "vllm_version": package_version("vllm"),
        "aiohttp_version": aiohttp.__version__,
        "commit": git_commit(),
    }
    return info, handle


def ensure_summary(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists() or path.stat().st_size == 0:
        path.write_text(",".join(SUMMARY_COLUMNS) + "\n", encoding="utf-8")


async def iter_sse(response: aiohttp.ClientResponse) -> AsyncIterator[str]:
    """Yield `data:` payloads from an SSE response."""
    data_lines: list[str] = []

    async for raw_line in response.content:
        line = raw_line.decode("utf-8").rstrip("\r\n")

        if not line:
            if data_lines:
                yield "\n".join(data_lines)
                data_lines.clear()
            continue

        if line.startswith(":"):
            continue

        if line.startswith("data:"):
            data_lines.append(line[5:].lstrip())

    if data_lines:
        yield "\n".join(data_lines)


async def monitor_vram(handle, stop: asyncio.Event, interval: float) -> float:
    peak = 0.0

    while not stop.is_set():
        used = nvmlDeviceGetMemoryInfo(handle).used / 1024**2
        peak = max(peak, used)

        try:
            await asyncio.wait_for(stop.wait(), timeout=interval)
        except asyncio.TimeoutError:
            pass

    used = nvmlDeviceGetMemoryInfo(handle).used / 1024**2
    return round(max(peak, used), 2)


async def run_one(
    session: aiohttp.ClientSession,
    semaphore: asyncio.Semaphore,
    args: argparse.Namespace,
    env: dict,
    index: int,
) -> dict:
    queued_at = time.perf_counter()

    record = {
        "client_request_id": f"bench-{uuid.uuid4().hex[:12]}",
        "server_request_id": None,
        "request_index": index,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "lane": args.lane,
        "workload": args.workload,
        "concurrency": args.concurrency,
        "model": args.model,
        "temperature": args.temperature,
        "seed": args.seed,
        "success": False,
        "metrics_complete": False,
        "http_status": None,
        "error_type": None,
        "error_message": None,
        "client_queue_ms": None,
        "ttft_ms": None,
        "tpot_ms": None,
        "e2e_ms": None,
        "itl_ms": [],
        "input_tokens": None,
        "output_tokens": None,
        "generated_text": "",
        "server_metrics": None,
        **env,
    }

    payload = {
        "model": args.model,
        "messages": [{"role": "user", "content": args.prompt}],
        "temperature": args.temperature,
        "seed": args.seed,
        "max_completion_tokens": args.max_tokens,
        "n": 1,
        "stream": True,
        "stream_options": {"include_usage": True},
    }

    try:
        async with semaphore:
            record["client_queue_ms"] = round(
                (time.perf_counter() - queued_at) * 1000, 3
            )

            start = time.perf_counter()
            last_timestamp = start
            first_chunk = False
            generated: list[str] = []

            async with session.post(args.endpoint, json=payload) as response:
                record["http_status"] = response.status

                if response.status != 200:
                    record["error_type"] = f"http_{response.status}"
                    record["error_message"] = (await response.text())[:2000]
                    return record

                async for message in iter_sse(response):
                    if message == "[DONE]":
                        break

                    timestamp = time.perf_counter()
                    data = json.loads(message)

                    if record["server_request_id"] is None:
                        record["server_request_id"] = data.get("id")

                    choices = data.get("choices")
                    if choices:
                        content = (choices[0].get("delta") or {}).get("content")

                        if not first_chunk:
                            first_chunk = True
                            record["ttft_ms"] = round(
                                (timestamp - start) * 1000, 3
                            )
                        else:
                            record["itl_ms"].append(
                                round((timestamp - last_timestamp) * 1000, 3)
                            )

                        if isinstance(content, str):
                            generated.append(content)

                    usage = data.get("usage")
                    if usage:
                        record["input_tokens"] = usage.get("prompt_tokens")
                        record["output_tokens"] = usage.get("completion_tokens")
                        record["server_metrics"] = data.get("metrics")

                    # Match vLLM benchmark timing semantics:
                    # final usage event is part of end-to-end response latency.
                    last_timestamp = timestamp

            if not first_chunk:
                record["error_type"] = "no_choice_event"
                record["error_message"] = "No streamed choices event received."
                return record

            record["generated_text"] = "".join(generated)
            record["e2e_ms"] = round((last_timestamp - start) * 1000, 3)

            output_tokens = record["output_tokens"]
            if isinstance(output_tokens, int) and output_tokens > 1:
                record["tpot_ms"] = round(
                    (record["e2e_ms"] - record["ttft_ms"]) / (output_tokens - 1),
                    3,
                )

            record["success"] = True
            record["metrics_complete"] = (
                record["ttft_ms"] is not None
                and record["e2e_ms"] is not None
                and isinstance(output_tokens, int)
                and output_tokens >= 1
                and (output_tokens == 1 or record["tpot_ms"] is not None)
            )

    except (asyncio.TimeoutError, aiohttp.ClientError, json.JSONDecodeError) as exc:
        record["error_type"] = type(exc).__name__
        record["error_message"] = str(exc)
    except Exception as exc:
        record["error_type"] = type(exc).__name__
        record["error_message"] = str(exc)

    return record


async def run_requests(
    session: aiohttp.ClientSession,
    args: argparse.Namespace,
    env: dict,
    count: int,
    concurrency: int,
) -> list[dict]:
    semaphore = asyncio.Semaphore(concurrency)

    return await asyncio.gather(
        *[
            run_one(session, semaphore, args, env, index)
            for index in range(count)
        ]
    )


async def async_main(args: argparse.Namespace) -> int:
    env, gpu_handle = environment_info(args.gpu_index)
    ensure_summary(Path(args.summary_csv))

    if env["vllm_version"] != EXPECTED_VLLM:
        print(
            f"WARNING: expected vLLM {EXPECTED_VLLM}, "
            f"found {env['vllm_version']}"
        )

    timeout = aiohttp.ClientTimeout(total=args.timeout_s)
    headers = {"Content-Type": "application/json"}

    if args.api_key:
        headers["Authorization"] = f"Bearer {args.api_key}"

    try:
        async with aiohttp.ClientSession(timeout=timeout, headers=headers) as session:
            # Warm-up is not written to raw JSONL.
            for i in range(args.warmup_requests):
                warm = await run_requests(session, args, env, 1, 1)
                if not warm[0]["success"]:
                    print(f"Warm-up {i + 1} failed: {warm[0]['error_message']}")
                    return 4

            stop_monitor = asyncio.Event()
            monitor_task = asyncio.create_task(
                monitor_vram(
                    gpu_handle,
                    stop_monitor,
                    args.vram_sample_interval_s,
                )
            )

            start = time.perf_counter()
            records = await run_requests(
                session,
                args,
                env,
                args.num_requests,
                args.concurrency,
            )
            duration = time.perf_counter() - start

            stop_monitor.set()
            vram_peak_mb = await monitor_task

    finally:
        nvmlShutdown()

    run_id = f"run-{uuid.uuid4().hex[:12]}"
    successes = sum(record["success"] for record in records)
    complete = sum(record["metrics_complete"] for record in records)
    output_tokens = sum(
        record["output_tokens"]
        for record in records
        if isinstance(record["output_tokens"], int)
    )

    for record in records:
        record["run_id"] = run_id
        record["run_duration_s"] = round(duration, 6)
        record["run_vram_peak_mb"] = vram_peak_mb

    raw_path = Path(args.raw_out)
    raw_path.parent.mkdir(parents=True, exist_ok=True)

    with raw_path.open("w" if args.overwrite else "a", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")

    print("\n===== M0-03 BENCHMARK RESULT =====")
    print(f"run_id              : {run_id}")
    print(f"requests            : {len(records)}")
    print(f"successes           : {successes}")
    print(f"metrics_complete    : {complete}")
    print(f"concurrency         : {args.concurrency}")
    print(f"duration_s          : {duration:.3f}")
    print(f"request_throughput  : {successes / duration:.3f} req/s")
    print(f"output_throughput   : {output_tokens / duration:.3f} tok/s")
    print(f"run_vram_peak_mb    : {vram_peak_mb}")
    print(f"raw_jsonl           : {raw_path}")
    print("===================================")

    if successes != len(records):
        return 2
    if complete != len(records):
        return 3
    return 0


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="M0-03 vLLM benchmark harness")

    parser.add_argument(
        "--endpoint",
        default="http://127.0.0.1:8000/v1/chat/completions",
    )
    parser.add_argument("--model", required=True)
    parser.add_argument(
        "--prompt",
        default="Trả lời đúng một câu: Hà Nội là thủ đô của nước nào?",
    )
    parser.add_argument("--num-requests", type=int, default=10)
    parser.add_argument("--concurrency", type=int, default=1)
    parser.add_argument("--warmup-requests", type=int, default=3)
    parser.add_argument("--max-tokens", type=int, default=32)
    parser.add_argument("--temperature", type=float, default=0.0)
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument(
        "--lane",
        choices=["vanilla", "static", "adaptive"],
        default="vanilla",
    )
    parser.add_argument("--workload", default="smoke")
    parser.add_argument("--timeout-s", type=float, default=120)
    parser.add_argument("--api-key", default=os.getenv("OPENAI_API_KEY"))
    parser.add_argument("--gpu-index", type=int, default=0)
    parser.add_argument("--vram-sample-interval-s", type=float, default=0.05)
    parser.add_argument(
        "--raw-out",
        default="experiments/runs/raw_test.jsonl",
    )
    parser.add_argument(
        "--summary-csv",
        default="experiments/results/summary.csv",
    )
    parser.add_argument("--overwrite", action="store_true")

    args = parser.parse_args()

    if args.num_requests < 1:
        parser.error("--num-requests must be >= 1")
    if not 1 <= args.concurrency <= args.num_requests:
        parser.error("--concurrency must be between 1 and --num-requests")
    if args.warmup_requests < 0:
        parser.error("--warmup-requests must be >= 0")
    if args.max_tokens < 1:
        parser.error("--max-tokens must be >= 1")

    return args


if __name__ == "__main__":
    raise SystemExit(asyncio.run(async_main(parse_args())))

Writing /kaggle/working/experiments/benchmark_harness.py


In [35]:
!ls -lh /kaggle/working/experiments/benchmark_harness.py

-rw-r--r-- 1 root root 27K Sep 17 23:57 /kaggle/working/experiments/benchmark_harness.py


In [37]:
!python -m py_compile /kaggle/working/experiments/benchmark_harness.py

In [38]:
!python /kaggle/working/experiments/benchmark_harness.py \
  --model Qwen/Qwen2.5-7B-Instruct-AWQ \
  --num-requests 1 \
  --concurrency 1 \
  --warmup-requests 0 \
  --max-tokens 16 \
  --raw-out /kaggle/working/experiments/runs/raw_test.jsonl \
  --summary-csv /kaggle/working/experiments/results/summary.csv \
  --overwrite


===== M0-03 BENCHMARK RESULT =====
run_id              : run-ea3022b905d9
requests            : 1
successes           : 1
metrics_complete    : 1
concurrency         : 1
duration_s          : 0.386
request_throughput  : 2.590 req/s
output_throughput   : 28.487 tok/s
run_vram_peak_mb    : 13539.19
raw_jsonl           : /kaggle/working/experiments/runs/raw_test.jsonl
summary_template    : /kaggle/working/experiments/results/summary.csv


In [39]:
!cd /kaggle/working && zip -r experiments_m0_03.zip experiments/

  adding: experiments/ (stored 0%)
  adding: experiments/benchmark_harness.py (deflated 75%)
  adding: experiments/results/ (stored 0%)
  adding: experiments/results/summary.csv (deflated 25%)
  adding: experiments/runs/ (stored 0%)
  adding: experiments/runs/raw_test.jsonl (deflated 43%)
  adding: experiments/__pycache__/ (stored 0%)
  adding: experiments/__pycache__/benchmark_harness.cpython-312.pyc (deflated 50%)


In [48]:
from IPython.display import FileLink

FileLink("/kaggle/working/experiments_m0_03.zip")

/kaggle/working/experiments_m0_03.zip

In [47]:
!ls -lh /kaggle/working/experiments_m0_03.zip
!unzip -l /kaggle/working/experiments_m0_03.zip | head -n 30

-rw-r--r-- 1 root root 29K Sep 18 00:02 /kaggle/working/experiments_m0_03.zip
Archive:  /kaggle/working/experiments_m0_03.zip
  Length      Date    Time    Name
---------  ---------- -----   ----
        0  2026-09-18 00:02   experiments/
    26970  2026-09-17 23:57   experiments/benchmark_harness.py
        0  2026-09-17 23:58   experiments/results/
      156  2026-09-17 23:58   experiments/results/summary.csv
        0  2026-09-17 23:58   experiments/runs/
     1207  2026-09-17 23:58   experiments/runs/raw_test.jsonl
        0  2026-09-17 23:58   experiments/__pycache__/
    26958  2026-09-17 23:58   experiments/__pycache__/benchmark_harness.cpython-312.pyc
    29045  2026-09-18 00:02   experiments/vllm_server.log
---------                     -------
    84336                     9 files


In [42]:
!cp /kaggle/working/vllm_server.log \
    /kaggle/working/experiments/vllm_server.log

In [43]:
!cd /kaggle/working && \
  zip -r experiments_m0_03.zip experiments/

updating: experiments/ (stored 0%)
updating: experiments/benchmark_harness.py (deflated 75%)
updating: experiments/results/ (stored 0%)
updating: experiments/results/summary.csv (deflated 25%)
updating: experiments/runs/ (stored 0%)
updating: experiments/runs/raw_test.jsonl (deflated 43%)
updating: experiments/__pycache__/ (stored 0%)
updating: experiments/__pycache__/benchmark_harness.cpython-312.pyc (deflated 50%)
  adding: experiments/vllm_server.log (deflated 77%)


In [46]:
from IPython.display import FileLink

FileLink("/kaggle/working/experiments_m0_03.zip")

/kaggle/working/experiments_m0_03.zip

In [53]:
from kaggle_secrets import UserSecretsClient
import os
import subprocess
from pathlib import Path

user_secrets = UserSecretsClient()

token = user_secrets.get_secret("GH_PAT")

askpass = Path("/kaggle/working/git-askpass.sh")
askpass.write_text(
    """#!/bin/sh
case "$1" in
  *Username*) echo "x-access-token" ;;
  *Password*) echo "$GITHUB_TOKEN" ;;
esac
"""
)
askpass.chmod(0o700)

env = os.environ.copy()
env["GITHUB_TOKEN"] = token
env["GIT_ASKPASS"] = str(askpass)
env["GIT_TERMINAL_PROMPT"] = "0"

subprocess.run(
    [
        "git",
        "clone",
        "--branch",
        "feat/m0-03-benchmark-harness",
        "--single-branch",
        "https://github.com/theflyingkhui04/do-an-ptit-d22.git",
        "/kaggle/working/do-an-ptit-d22",
    ],
    env=env,
    check=True,
)

askpass.unlink()
print("Clone OK")

Cloning into '/kaggle/working/do-an-ptit-d22'...


Clone OK


In [54]:
!git -C /kaggle/working/do-an-ptit-d22 checkout --detach 397f48eb9b928746c7cb2e8464e6bb9212f00754

!git -C /kaggle/working/do-an-ptit-d22 rev-parse HEAD

HEAD is now at 397f48e feat: add M0-03 benchmark harness
397f48eb9b928746c7cb2e8464e6bb9212f00754


In [81]:
!curl -sS -o /dev/null -w "%{http_code}\n" \
  http://127.0.0.1:8000/health

200


In [82]:
!cd /kaggle/working/do-an-ptit-d22 && \
python -m py_compile experiments/benchmark_harness.py

In [83]:
!cd /kaggle/working/do-an-ptit-d22 && \
git status --short

In [84]:
!cd /kaggle/working/do-an-ptit-d22 && \
python experiments/benchmark_harness.py \
  --model Qwen/Qwen2.5-7B-Instruct-AWQ \
  --num-requests 1 \
  --concurrency 1 \
  --warmup-requests 0 \
  --max-tokens 16 \
  --overwrite


===== M0-03 BENCHMARK RESULT =====
run_id              : run-14ca0309d6fb
requests            : 1
successes           : 1
metrics_complete    : 1
concurrency         : 1
duration_s          : 0.430
request_throughput  : 2.328 req/s
output_throughput   : 25.605 tok/s
run_vram_peak_mb    : 14343.19
raw_jsonl           : experiments/runs/raw_test.jsonl


In [85]:
import json

path = "/kaggle/working/do-an-ptit-d22/experiments/runs/raw_test.jsonl"

with open(path, encoding="utf-8") as f:
    r = json.loads(f.readline())

for key in [
    "success",
    "metrics_complete",
    "ttft_ms",
    "tpot_ms",
    "e2e_ms",
    "input_tokens",
    "output_tokens",
    "gpu",
    "vllm_version",
    "commit",
]:
    print(f"{key:18}: {r.get(key)}")

success           : True
metrics_complete  : True
ttft_ms           : 151.24
tpot_ms           : 27.801
e2e_ms            : 429.246
input_tokens      : 45
output_tokens     : 11
gpu               : Tesla T4
vllm_version      : 0.29.0
commit            : 397f48eb9b928746c7cb2e8464e6bb9212f00754


In [86]:
!cd /kaggle/working/do-an-ptit-d22 && \
python experiments/benchmark_harness.py \
  --model Qwen/Qwen2.5-7B-Instruct-AWQ \
  --num-requests 10 \
  --concurrency 4 \
  --warmup-requests 3 \
  --max-tokens 32 \
  --lane vanilla \
  --workload smoke \
  --overwrite


===== M0-03 BENCHMARK RESULT =====
run_id              : run-b9a171b40abf
requests            : 10
successes           : 10
metrics_complete    : 10
concurrency         : 4
duration_s          : 0.967
request_throughput  : 10.341 req/s
output_throughput   : 113.751 tok/s
run_vram_peak_mb    : 14343.19
raw_jsonl           : experiments/runs/raw_test.jsonl


In [87]:
!wc -l /kaggle/working/do-an-ptit-d22/experiments/runs/raw_test.jsonl

10 /kaggle/working/do-an-ptit-d22/experiments/runs/raw_test.jsonl


In [88]:
import json

path = "/kaggle/working/do-an-ptit-d22/experiments/runs/raw_test.jsonl"

with open(path, encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]

print("rows             :", len(rows))
print("successes        :", sum(r["success"] for r in rows))
print("metrics_complete :", sum(r["metrics_complete"] for r in rows))
print("HTTP 200         :", sum(r["http_status"] == 200 for r in rows))
print("commit unique    :", {r["commit"] for r in rows})
print("concurrency      :", {r["concurrency"] for r in rows})
print("GPU              :", {r["gpu"] for r in rows})

assert len(rows) >= 10
assert all(r["success"] for r in rows)
assert all(r["metrics_complete"] for r in rows)
assert all(r["http_status"] == 200 for r in rows)
assert all(
    r["commit"] == "397f48eb9b928746c7cb2e8464e6bb9212f00754"
    for r in rows
)

print("M0-03 concurrent acceptance: PASS")

rows             : 10
successes        : 10
metrics_complete : 10
HTTP 200         : 10
commit unique    : {'397f48eb9b928746c7cb2e8464e6bb9212f00754'}
concurrency      : {4}
GPU              : {'Tesla T4'}
M0-03 concurrent acceptance: PASS
